In [0]:
%sql
DECLARE OR REPLACE VARIABLE profiling_sql STRING;

SET VAR profiling_sql = (
    SELECT concat(
        'CREATE OR REPLACE TABLE profile_source_data AS ',
        concat_ws(
        '\nUNION ALL\n',

        collect_list(
            concat(
                'SELECT ',
                chr(39), table_name, chr(39), ' AS table_name, ',
                chr(39), column_name, chr(39), ' AS column_name, ',
                chr(39), data_type, chr(39), ' AS data_type, ',

                'COUNT(*) AS total_rows, ',

                'COUNT(*) - COUNT(`', column_name, '`) AS null_count, ',

                'ROUND(100.0 * ',
                '(COUNT(*) - COUNT(`', column_name, '`)) / COUNT(*), 2',
                ') AS null_percentage, ',

                'COUNT(DISTINCT `', column_name, '`) AS distinct_count, ',

                'ROUND(100.0 * COUNT(DISTINCT `', column_name, '`) ',
                '/ COUNT(*), 2) AS distinct_percentage, ',

                'CAST(MIN(`', column_name, '`) AS STRING) AS min_value, ',

                'CAST(MAX(`', column_name, '`) AS STRING) AS max_value, ',

                '(SELECT CAST(`', column_name, '` AS STRING) ',
                'FROM `nyc_mobility`.`raw`.`', table_name, '` ',
                'WHERE `', column_name, '` IS NOT NULL ',
                'GROUP BY `', column_name, '` ',
                'ORDER BY COUNT(*) DESC, `', column_name, '` ',
                'LIMIT 1) AS most_common_value ',

                'FROM `nyc_mobility`.`raw`.`', table_name, '`'
            )
        )
    )
    )
    FROM nyc_mobility.information_schema.columns
    WHERE table_schema = 'raw'
);

EXECUTE IMMEDIATE profiling_sql;